**Step 0: Check if DEM & ACC have the same CRS with 186 Watershed boundaries and convert if needed**

In [23]:
from pathlib import Path
import os
os.environ["GEOPANDAS_IO_ENGINE"] = "pyogrio"   
import geopandas as gpd
import rasterio   #This is a library for working with raster geospatial data (GeoTIFF)
import rioxarray as rxr # extension of xarray + rasterio, simplifies reading/writing GeoTIFF and reprojection to different CRS


acc_path = Path("../../data/HydroSHEDS/as_acc_Bhutan_and_buffer.tif") #ACC data from HydroSHEDS
dem_path = Path("../../data/HydroSHEDS/as_dem_Bhutan_and_buffer.tif") #DEM data from HydroSHEDS
polygon_coordinates_path = Path("../../data/boundaries/186_watershed/186 Watershed boundary.shp") #186 Watershed boundary.shp → geometry (polygon coordinates)
indexes_path = Path("../../data/boundaries/186_watershed/186 Watershed boundary.shx") #indexes for quick access
attribute_path = Path("../../data/boundaries/186_watershed/186 Watershed boundary.dbf") #attribute table (ID, other fields)
coordinate_system_path = Path("../../data/boundaries/186_watershed/186 Watershed boundary.prj") #To align Coordinate Reference System with DEM

if not (acc_path.exists() == dem_path.exists() == polygon_coordinates_path.exists() == indexes_path.exists() == attribute_path.exists() == coordinate_system_path.exists() ==True):
    print ("sushi")
    raise FileNotFoundError("One or more required files are missing. Please check the paths.")

# -----Check if DEM, ACC and 186_Watershed have the same CRS -----
ws = gpd.read_file(polygon_coordinates_path, engine="pyogrio")
             
with rasterio.open(dem_path) as r:
    dem_crs = r.crs
    dem_nodata = r.nodata

with rasterio.open(acc_path) as ra:
    acc_crs = ra.crs
    acc_nodata = ra.nodata

if not ws.crs == dem_crs==acc_crs:
    print("CRS NOT match!")
    print("WS CRS:", ws.crs)
    print("DEM CRS:", dem_crs)
    print("ACC CRS:", acc_crs)
    print("Let's bring all datasets to the same CRS:")

    # Covert DEM to the same CRS as the watershed shapefile (EPSG:3857)
    dem = rxr.open_rasterio(dem_path)
    dem_reproj = dem.rio.reproject(ws.crs)  # Reproject DEM to match watershed CRS
    dem_out = Path("../../data/HydroSHEDS/dem_Bhutan_and_buffer_EPSG3857.tif")
    dem_reproj.rio.to_raster(dem_out)

    # Covert ACC to the same CRS as the watershed shapefile (EPSG:3857)
    acc = rxr.open_rasterio(acc_path)
    acc_reproj = acc.rio.reproject(ws.crs)
    acc_out = Path("../../data/HydroSHEDS/acc_Bhutan_and_buffer_EPSG3857.tif")
    acc_reproj.rio.to_raster(acc_out)

    print(f"DEM saved to {dem_out}")
    print(f"ACC saved to {acc_out}")
else:
    print("All datasets already share the same CRS.")

CRS NOT match!
WS CRS: EPSG:3857
DEM CRS: EPSG:4326
ACC CRS: EPSG:4326
Let's bring all datasets to the same CRS:
DEM saved to ../../data/HydroSHEDS/dem_Bhutan_and_buffer_EPSG3857.tif
ACC saved to ../../data/HydroSHEDS/acc_Bhutan_and_buffer_EPSG3857.tif


**Select unique watershed identifier**

In [24]:
import geopandas as gpd
from pathlib import Path

ws_path = Path("../../data/boundaries/186_watershed/186 Watershed boundary.shp")

# load shapefile
ws = gpd.read_file(ws_path, engine="pyogrio")
print(ws.head())

# list all available columns
print("\nColumns in shapefile:", ws.columns.tolist())

         f_area  ws_id           area_ha     area_ha_1     SHAPE__Len  \
0  1.403314e+08      1  14033.1356069597  14033.135607   88962.631299   
1  1.069433e+08      2   10694.331035308  10694.331035   72762.793824   
2  1.634449e+08      3  16436.0007344264  16436.000734   87853.120695   
3  2.076810e+08      4  20768.1036079112  20768.103608  100646.751220   
4  9.158350e+07      5  9158.35047410094   9158.350474   62076.446669   

     SHAPE__Are                                           geometry  
0  1.783218e+08  POLYGON ((10245897.804 3155396.599, 10245640.9...  
1  1.380322e+08  POLYGON ((9990796.024 3272558.723, 9990916.261...  
2  2.119142e+08  POLYGON ((9977078.645 3269692.821, 9977264.969...  
3  2.681361e+08  POLYGON ((9997299.005 3278803.177, 9997517.596...  
4  1.181974e+08  POLYGON ((10034395.122 3273465.314, 10034625.4...  

Columns in shapefile: ['f_area', 'ws_id', 'area_ha', 'area_ha_1', 'SHAPE__Len', 'SHAPE__Are', 'geometry']


**Compute zonal statistics for each watershed (ws_id)** 

In [25]:
from pathlib import Path
import pandas as pd
import os
os.environ["GEOPANDAS_IO_ENGINE"] = "pyogrio"   
import geopandas as gpd
from rasterstats import zonal_stats #It allows to calculate statistics on raster values (for example, a DEM) within vector polygons


dem_reproj = Path("../../data/HydroSHEDS/dem_Bhutan_and_buffer_EPSG3857.tif")
acc_reproj = Path("../../data/HydroSHEDS/acc_Bhutan_and_buffer_EPSG3857.tif")
ws_path = Path("../../data/boundaries/186_watershed/186 Watershed boundary.shp")
# When reading a shapefile we specify only the .shp file, 
# but geopandas automatically also loads the companion files:
# - .shp → geometry
# - .dbf → attributes
# - .shx → index
# - .prj → CRS

ws = gpd.read_file(ws_path, engine="pyogrio")

# zonal_stats computes statistics for raster values within each vector polygon
#
# Commonly used stats keywords:
#   - "min"        → minimum value
#   - "max"        → maximum value
#   - "mean"       → arithmetic mean
#   - "median"     → median value
#   - "std"        → standard deviation
#   - "sum"        → sum of all valid cell values
#   - "count"      → number of valid cells
#   - "range"      → difference between max and min
#   - "majority"   → most frequent value
#   - "minority"   → least frequent value
#   - "unique"     → list of unique values
#   - "all"        → compute all available statistics
#   - "percentile_X" → e.g. "percentile_90" = 90th percentile
#
# nodata=-9999 tells the function to ignore raster cells with value -9999
# (these represent missing data / outside valid area in many DEM/ACC files)

# DEM zonal statistics
dem_stats = zonal_stats(ws, dem_reproj, stats=["min", "max", "mean", "median", "std"], nodata=-9999)
# ACC zonal statistics
acc_stats = zonal_stats(ws, acc_reproj, stats=["min", "max", "mean", "median", "std"], nodata=-9999)

# Convert lists of dicts to DataFrames and prefix columns
df_dem = pd.DataFrame(dem_stats).add_prefix("dem_")
df_acc = pd.DataFrame(acc_stats).add_prefix("acc_")

# Combine with ws_id
result = ws[["ws_id"]].copy()
result = pd.concat([ws["ws_id"].reset_index(drop=True), df_dem, df_acc], axis=1)

# Save to CSV
out_csv = ws_path.parent / "watershed_dem_acc_stats.csv"
num_records = len(result)
print(f"Number of records in result: {num_records}")
print (result.head())
result.to_csv(out_csv, index=False)
print(f"Statistics saved to {out_csv.resolve()}")

Number of records in result: 186
   ws_id  dem_min  dem_max     dem_mean     dem_std  dem_median  acc_min  \
0      1   2690.0   4483.0  3871.240740  350.309780      3952.0      1.0   
1      2   3433.0   6141.0  4576.269356  479.646825      4593.0      1.0   
2      3   3409.0   6447.0  4636.337642  468.489801      4651.0      1.0   
3      4   3395.0   6209.0  4970.112385  500.694309      5033.0      1.0   
4      5   3885.0   7026.0  4984.136151  702.877441      4817.0      1.0   

   acc_max    acc_mean      acc_std  acc_median  
0   8601.0   77.982920   564.187823         3.0  
1  36739.0  113.261979  1023.329133         4.0  
2  36822.0  155.080198  1498.649756         4.0  
3  36840.0  171.237076  1710.621052         4.0  
4  10835.0  121.560912   762.528089         4.0  
Statistics saved to /home/merlin/bhutan_climate_modeling/data/boundaries/186_watershed/watershed_dem_acc_stats.csv


**Select unique basin identifier**

In [26]:
from pathlib import Path

ws_path = Path("../../data/boundaries/basins/Basin boundary.shp")

# load shapefile
ws = gpd.read_file(ws_path, engine="pyogrio")
print(ws.head())

# list all available columns
print("\nColumns in shapefile:", ws.columns.tolist())

           basin                                             path  \
0        Aiechhu        E:\2024-2025\Basin_boundaries\Aiechhu.shp   
1  Merak_Sakteng  E:\2024-2025\Basin_boundaries\Merak-Sakteng.shp   
2     Mangdechhu      E:\2024-2025\Basin_boundaries\Tb_basins.shp   
3      Jaldhakha      E:\2024-2025\Basin_boundaries\Tb_basins.shp   
4        Amochhu      E:\2024-2025\Basin_boundaries\Tb_basins.shp   

         area_ha    area_sqkm     SHAPE__Len    SHAPE__Are  \
0  196390.588555  1963.905886  337019.366034  2.481073e+09   
1   13881.067718   138.810677   76767.739616  1.763939e+08   
2  743660.531079  7436.605311  524792.855796  9.488740e+09   
3  103894.293085  1038.942931  252015.441762  1.314962e+09   
4  392961.936740  3929.619367  488429.626674  4.999591e+09   

                                            geometry  
0  POLYGON ((10058028.821 3151062.425, 10058067.6...  
1  POLYGON ((10245694.293 3155611.158, 10245655.4...  
2  POLYGON ((10104749.232 3241419.811, 10104770

**Compute zonal statistics for each basins (basin)** 

In [27]:
from pathlib import Path
import pandas as pd
import os
os.environ["GEOPANDAS_IO_ENGINE"] = "pyogrio"   
import geopandas as gpd
from rasterstats import zonal_stats #It allows to calculate statistics on raster values (for example, a DEM) within vector polygons


dem_reproj = Path("../../data/HydroSHEDS/dem_Bhutan_and_buffer_EPSG3857.tif")
acc_reproj = Path("../../data/HydroSHEDS/acc_Bhutan_and_buffer_EPSG3857.tif")
ws_path = Path("../../data/boundaries/basins/Basin boundary.shp")
# When reading a shapefile we specify only the .shp file, 
# but geopandas automatically also loads the companion files:
# - .shp → geometry
# - .dbf → attributes
# - .shx → index
# - .prj → CRS

ws = gpd.read_file(ws_path, engine="pyogrio")

# zonal_stats computes statistics for raster values within each vector polygon
#
# Commonly used stats keywords:
#   - "min"        → minimum value
#   - "max"        → maximum value
#   - "mean"       → arithmetic mean
#   - "median"     → median value
#   - "std"        → standard deviation
#   - "sum"        → sum of all valid cell values
#   - "count"      → number of valid cells
#   - "range"      → difference between max and min
#   - "majority"   → most frequent value
#   - "minority"   → least frequent value
#   - "unique"     → list of unique values
#   - "all"        → compute all available statistics
#   - "percentile_X" → e.g. "percentile_90" = 90th percentile
#
# nodata=-9999 tells the function to ignore raster cells with value -9999
# (these represent missing data / outside valid area in many DEM/ACC files)

# DEM zonal statistics
dem_stats = zonal_stats(ws, dem_reproj, stats=["min", "max", "mean", "median", "std"], nodata=-9999)
# ACC zonal statistics
acc_stats = zonal_stats(ws, acc_reproj, stats=["min", "max", "mean", "median", "std"], nodata=-9999)

# Convert lists of dicts to DataFrames and prefix columns
df_dem = pd.DataFrame(dem_stats).add_prefix("dem_")
df_acc = pd.DataFrame(acc_stats).add_prefix("acc_")

# Combine with ws_id
result = ws[["basin"]].copy()
result = pd.concat([ws["basin"].reset_index(drop=True), df_dem, df_acc], axis=1)

# Save to CSV
out_csv = ws_path.parent / "basin_dem_acc_stats.csv"
num_records = len(result)
print(f"Number of records in result: {num_records}")
print (result.head())
result.to_csv(out_csv, index=False)
print(f"Statistics saved to {out_csv.resolve()}")

Number of records in result: 10
           basin  dem_min  dem_max     dem_mean      dem_std  dem_median  \
0        Aiechhu     92.0   4160.0  1183.538707   699.525166      1091.0   
1  Merak_Sakteng   2690.0   4483.0  3882.785851   340.278285      3965.0   
2     Mangdechhu    109.0   7065.0  3230.588248  1302.878378      3287.0   
3      Jaldhakha    210.0   4577.0  1588.797296  1011.341252      1403.0   
4        Amochhu    164.0   6689.0  3187.160344  1411.272228      3521.0   

   acc_min    acc_max     acc_mean       acc_std  acc_median  
0      1.0  3754703.0   210.078824   7825.328906         3.0  
1      1.0    10252.0    65.099067    487.918880         3.0  
2      1.0   979449.0  1070.502172  21801.824509         3.0  
3      1.0    72476.0   199.115628   2573.336117         3.0  
4      1.0   497461.0   894.656866  16011.294545         3.0  
Statistics saved to /home/merlin/bhutan_climate_modeling/data/boundaries/basins/basin_dem_acc_stats.csv
